# 01 - Exploratory Data Analysis (EDA)
**Person A: Data, Weather, and Features**

### Objectives:
1. Load the raw Kaggle dataset and examine the schema.
2. Determine how the target is defined (joint single target with `Source` column).
3. Clean the dataset (handle timestamps, verify nulls and duplicates).
4. Analyze daily, weekly, and seasonal patterns in energy production.
5. Contrast Wind vs. Solar distribution and generation share.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_loading import load_raw, clean
from plotting import apply_style, savefig, COLORS

apply_style()

## 1. Load and Clean Dataset

In [ ]:
df_raw = load_raw()
print(f"Raw shape: {df_raw.shape}")
df = clean(df_raw)
print(f"Cleaned shape: {df.shape}")
print(f"Date range: {df.index.min()} -> {df.index.max()}")
print(f"Source breakdown:\n{df['Source'].value_counts()}")
df.head()

## 2. Production Distributions by Energy Source

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, src in zip(axes, ['Wind', 'Solar']):
    sub = df[df['Source'] == src]['Production']
    ax.hist(sub, bins=40, color=COLORS[src], edgecolor='white', alpha=0.85)
    ax.set_title(f'{src} Production Distribution')
    ax.set_xlabel('Production (MWh)')
    ax.set_ylabel('Frequency')
plt.tight_layout()
savefig('../results/figures/01_distribution.png')
plt.show()

## 3. Average Diurnal Cycle (Hourly Profile)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for src in ['Wind', 'Solar']:
    sub = df[df['Source'] == src]
    hourly = sub.groupby(sub.index.hour)['Production'].mean()
    ax.plot(hourly.index, hourly.values, marker='o', label=src, color=COLORS[src])
ax.set_title('Average Hourly Production Profile (Diurnal Cycle)')
ax.set_xlabel('Hour of Day (0-23)')
ax.set_ylabel('Mean Production')
ax.set_xticks(range(0, 24, 2))
ax.legend()
plt.tight_layout()
savefig('../results/figures/01_hourly_avg.png')
plt.show()

## 4. Monthly & Long-Term Trends (2020 - 2025)

In [ ]:
df_monthly = df.copy()
df_monthly['year_month'] = df_monthly.index.to_period('M')
monthly = df_monthly.groupby(['year_month', 'Source'])['Production'].mean().reset_index()
monthly['date'] = monthly['year_month'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(12, 4.5))
for src in ['Wind', 'Solar']:
    sub = monthly[monthly['Source'] == src]
    ax.plot(sub['date'], sub['Production'], label=src, color=COLORS[src], marker='.')
ax.set_title('Monthly Average Production (2020 - 2025)')
ax.set_xlabel('Date')
ax.set_ylabel('Mean Production')
ax.legend()
plt.tight_layout()
savefig('../results/figures/01_monthly.png')
plt.show()

## 5. Seasonal Patterns

In [ ]:
def get_season(m):
    if m in [3, 4, 5]: return 'Spring'
    elif m in [6, 7, 8]: return 'Summer'
    elif m in [9, 10, 11]: return 'Fall'
    else: return 'Winter'

df['season'] = df.index.month.map(get_season)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
season_order = ['Spring', 'Summer', 'Fall', 'Winter']
for ax, src in zip(axes, ['Wind', 'Solar']):
    sub = df[df['Source'] == src]
    sns.boxplot(data=sub, x='season', y='Production', hue='season', order=season_order, ax=ax, 
                palette='Blues' if src=='Wind' else 'YlOrBr', legend=False)
    ax.set_title(f'{src} Production by Season')
    ax.set_xlabel('Season')
    ax.set_ylabel('Production')
plt.tight_layout()
savefig('../results/figures/01_seasonal_box.png')
plt.show()

## 6. Energy Source Production Share

In [ ]:
share = df[df['Source'].isin(['Wind', 'Solar'])].groupby('Source')['Production'].sum()
fig, ax = plt.subplots(figsize=(6, 5))
ax.pie(share, labels=share.index, autopct='%1.1f%%', colors=[COLORS['Solar'], COLORS['Wind']], startangle=90, explode=(0.05, 0))
ax.set_title('Total Production Share: Wind vs Solar')
plt.tight_layout()
savefig('../results/figures/01_source_share.png')
plt.show()